# 05. 리텐션 캠페인 A/B 사전 설계

REES46 공개 로그로 리마인드 실험을 설계했다. 사전 입력은 **3/26 UTC 하루 종료까지**, 결과 관찰은 **3/27~4/30(35일)**이다. 04에서 라벨을 생성하고, 별도로 원본에서 생성한 Recency 후보와 사전 구매 캐시를 검토 기록 및 파일 해시와 대조한 뒤 사용한다.

## Recency 구간별 관찰 기간 활동

대상은 3/26까지 한 번 이상 관측된 사용자다. Recency는 3/26에서 마지막 사전 활동의 UTC 날짜를 뺀 일수다. 활동률은 구간·층별 대상 중 관찰 기간에 한 번 이상 활동한 비율이다. 관찰 기간에 처음 관측된 사용자는 판단 보류로 분리한다.


In [1]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import config
import pandas as pd
from IPython.display import Markdown, display
from src.ab_design import summarize_risk, summary_markdown

from src.recency_segments import load_comparison, with_totals, comparison_markdown, check_risk_alignment

strata, risk = load_comparison()
comparison = with_totals(strata)
summary = summarize_risk(risk)
check_risk_alignment(comparison, summary)
comparison.to_csv(config.RECENCY_COMPARISON_PATH, index=False)
display(Markdown(comparison_markdown(comparison)))

| Recency(일) | 구매층 | 대상자 수 | 관찰 기간 활동자 수 | 관찰 기간 활동률 |
|---|---|---:|---:|---:|
| 0-7 | 전체 | 1,158,712 | 680,047 | 58.69% |
| 0-7 | 사전 구매 | 327,485 | 250,358 | 76.45% |
| 0-7 | 사전 비구매 | 831,227 | 429,689 | 51.69% |
| 8-13 | 전체 | 634,624 | 269,894 | 42.53% |
| 8-13 | 사전 구매 | 152,948 | 89,205 | 58.32% |
| 8-13 | 사전 비구매 | 481,676 | 180,689 | 37.51% |
| 14-34 | 전체 | 2,852,344 | 726,061 | 25.45% |
| 14-34 | 사전 구매 | 376,467 | 161,770 | 42.97% |
| 14-34 | 사전 비구매 | 2,475,877 | 564,291 | 22.79% |
| 35+ (휴면 기준 충족군의 복귀) | 전체 | 8,702,296 | 833,164 | 9.57% |
| 35+ (휴면 기준 충족군의 복귀) | 사전 구매 | 854,670 | 112,131 | 13.12% |
| 35+ (휴면 기준 충족군의 복귀) | 사전 비구매 | 7,847,626 | 721,033 | 9.19% |

## 14~34일을 설계 대상으로 두는 근거

상한은 전체 7개월의 관측 재방문 간격 p90을 참고해 확정한 35일 기준 미만으로 둔다. 하한은 사후 정보를 사용하지 않도록 기준일 이전 분포의 p75 + 1일로 정한다. 전체 기간과 사전 기간의 분위수를 나란히 보고하며, p75·p90의 선택은 운영 기준이고 최적값으로 검증한 것은 아니다.

**하한은 기준일 이전 완료 간격 p75(13일)에 1일을 더한 운영 기준.** 사전 분포의 p75·p90은 13·31일, 전체 분포는 14·35일이다. 사전 기간은 관측 창이 짧아 긴 간격이 덜 관측될 수 있다. 두 수치는 완료 간격 기준이다.

14~34일 구간은 위 표에서 앞선 구간보다 관측 활동률이 낮고, 설계 표본보다 대상 풀이 크다. 수신 동의와 실제 도달 가능 규모는 확인되지 않았다. 35+는 기준 시점 휴면 기준 충족군이며 사후 미활동 라벨과는 관측 시점이 다르다.

[일수별 재방문율](../reports/recency_day_outcome_comparison.csv)은 관찰 비교다. 28일에 해당하는 2/27의 원본 활동 기록은 인접일보다 적었고 원인은 미확인이다. 구간 간 활동률 차이로 캠페인 효과나 최적 하한을 추정하지 않는다. 전체 활동률은 두 층의 분자·분모를 더해 계산한다.

## 위험군 baseline과 표본 설계


In [2]:
display(Markdown(summary_markdown(summary)))

| 구분 | 대상자 수 | 관찰 기간 재방문자 수 | baseline | 설계 대립값(+5%p) | 군당 표본 |
|---|---:|---:|---:|---:|---:|
| 전체 위험군 | 2,852,344 | 726,061 | 25.45% | 30.45% | 1,263 |
| 기준일 이전 구매 경험군 | 376,467 | 161,770 | 42.97% | 47.97% | 1,556 |
| 기준일 이전 비구매군 | 2,475,877 | 564,291 | 22.79% | 27.79% | 1,184 |

## 입력 시점과 층화

`is_buyer_at_reference`는 3/26 종료까지의 관측 구매 여부다. 04의 7개월 전체 `is_buyer`는 사후 기술통계이며 층화에는 사전 구매 캐시를 사용한다.


In [3]:
stored = pd.read_csv(config.RUN_REPORTS_DIR / "ab_design_summary.csv")
pd.testing.assert_frame_equal(summary, stored, check_exact=False, rtol=1e-12)
assert summary.iloc[1:]["users"].sum() == summary.iloc[0]["users"]
assert summary.iloc[1:]["returned_users"].sum() == summary.iloc[0]["returned_users"]
assert risk["status"].isin(["retained", "churned"]).all()
assert risk["recency_at_ref"].between(*config.risk_bounds()).all()
print("저장 CSV 일치 · 층별 인원/재방문자 합계 일치 · 위험군/라벨 검사 통과")

저장 CSV 일치 · 층별 인원/재방문자 합계 일치 · 위험군/라벨 검사 통과


## 표본과 실행 설계

alpha 0.05, power 0.80, 절대 MDE +5%p, 양측 검정, 처치·대조 1:1이다. Cohen’s h를 효과크기로 사용한 statsmodels `NormalIndPower`의 결과를 군당 표본 수로 올림한다.

[가정한 절대 재방문율 차이에 따른 군당 필요 표본 수](../docs/ab_test_design.md#가정한-절대-재방문율-차이에-따른-군당-필요-표본-수)

전체를 1차 판단, 층별을 탐색 분석으로 둔다. 대상 풀의 층 비중을 유지해 추출한 뒤 각 층 내에서 1:1 배정한다. 전체 표본 확보는 각 층의 검정력 확보와 다르며, 층마다 같은 인원을 뽑으면 전체 효과의 모집단 가중치를 다시 정해야 한다.

Primary는 전체 배정 사용자를 분모로 한 35일 재방문율(ITT). Secondary는 구매율·객단가이며 주문 ID가 없어 객단가는 운영 데이터 확보 후 산식을 정한다. Guardrail은 수신거부·비용·마진이며 실행 전에 허용 악화폭을 정한다. MDE는 검정력 설계 조건으로, 자동 성공 기준은 아니다. SRM·중복 배정·관찰 누락을 확인하고 임의 조기 중단을 피한다.

실제 실험을 집행하지 않았으며 수신 동의·비용 정보가 없어 바로 집행 가능한 운영 명세는 아니다. 자세한 산식과 제한은 [A/B 설계서](../docs/ab_test_design.md)를 따른다.

In [4]:
import json
audit = json.loads((config.RUN_REPORTS_DIR / "ab_design_validation.json").read_text())
print("기준일:", audit["reference_date"], audit["timezone"])
print("사전 입력 종료(미포함):", audit["feature_end_exclusive"])
print("결과 관찰 종료(미포함):", audit["outcome_end_exclusive"])
print("사전 구매 경험자:", f"{audit['pre_reference_buyers_all_users']:,}명")
print("전체 기간 구매 피처를 대입할 때 층이 달라질 위험군:", f"{audit['reclassified_future_only_buyers']:,}명")


기준일: 2020-03-26 UTC
사전 입력 종료(미포함): 2020-03-27T00:00:00Z
결과 관찰 종료(미포함): 2020-05-01T00:00:00Z
사전 구매 경험자: 1,711,570명
전체 기간 구매 피처를 대입할 때 층이 달라질 위험군: 43,919명
